In [13]:
import pandas as pd
city_df = pd.read_csv("city.csv")
def get_city_code(city_name: str) -> int:
    """
    获取城市编码
    :param city_name:
    :return: 城市编码
    """
    match = city_df[city_df['district'] == city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    match = city_df[city_df['city'] == city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    match = city_df[city_df['city'].str.contains(city_name, na=False)]
    if not match.empty:
         return match.iloc[0]['areacode/城市ID']
    return 101010100


city_df.head(5)


,areacode/城市ID,province_geocode,province,city_geocode,city,district_geocode,district,eng,pinyin,lon,lat,exclude（是否为地级市）
0,101010900,110000,北京市,110100,北京市,110106,丰台,fengtai,fengtai,116.286968,39.863642,0
1,101011000,110000,北京市,110100,北京市,110107,石景山,shijingshan,shijingshan,116.195445,39.914601,0
2,101011400,110000,北京市,110100,北京市,110109,门头沟,mentougou,mentougou,116.105381,39.937183,0
3,101011200,110000,北京市,110100,北京市,110111,房山,fangshan,fangshan,116.139157,39.735535,0
4,101010600,110000,北京市,110100,北京市,110112,通州,tongzhou,tongzhou,116.658603,39.902486,0


In [23]:
import requests

@tool
def get_weather(city: str) -> str:
    """
    查询实时天气API，返回温度以及天气情况
    :param city:城市名称，例如:北京
    :return:温度以及天气情况
    """
    city_code = get_city_code(city)
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    payload = {"areacode" : city_code}
    headers = {
        "X-APISpace-Token":"idouvpjpwppkyibjrly12yo4vse8mdpy"
    }
    response=requests.request("GET", url, params=payload, headers=headers)
    data = response.json()
    tmp = data.get('result').get('realtime').get('temp')
    wd = data.get('result').get('realtime').get('text')

    return f'天气: {wd},温度: {tmp} ℃'


#get_weather("石景山")



### 调用工具

In [24]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [33]:
from langchain_core.tools import tool
import re

@tool
def multiply(input_str: str) -> int:
    """计算两个整数的乘积。"""

    # r'\d+' 是一个正则表达式：
    #
    # \d  ：匹配一个数字，例如 0~9
    # +   ：表示前面的内容出现 1 次或多次
    # re.findall() 会找出字符串中所有符合条件的内容，
    # 并以列表形式返回。
    #
    # 例如：
    # input_str = "5, 6"
    # match = ["5", "6"]
    match = re.findall(r'\d+', input_str)
    if len(match) == 2:
        # map(int, match)
        # 相当于：
        #
        # int("5") -> 5
        # int("6") -> 6
        a, b = map(int, match)
    return a * b

print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

multiply
计算两个整数的乘积。
{'input_str': {'title': 'Input Str', 'type': 'string'}}
{'description': '计算两个整数的乘积。', 'properties': {'input_str': {'title': 'Input Str', 'type': 'string'}}, 'required': ['input_str'], 'title': 'multiply', 'type': 'object'}


In [36]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
#创建工具列表
tools = [get_weather, multiply]

#获取提示词
#prompt = hub.pull("hwchase17/react",dangerously_pull_public_prompt=True)
template = template = """
你是一个可以使用工具解决问题的助手。

你可以使用以下工具：

{tools}

可用工具名称：
{tool_names}

你必须严格遵循下面的格式。

如果需要调用工具：

Question: 用户的问题
Thought: 思考应该使用什么工具
Action: 必须是 [{tool_names}] 中的一个
Action Input: 提供给工具的输入
Observation: 工具返回结果

如果已经获得答案，或者问题不需要调用任何工具，必须使用：

Thought: 我已经知道答案
Final Answer: 最终回答

注意：
1. 不允许直接输出普通答案。
2. 最终回答必须以 "Final Answer:" 开头。
3. 如果不需要工具，也必须输出 "Final Answer:"。

Question: {input}

Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)
#创建agent
agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)

#创建AgentExecutor 运行agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True) # verbose是输出日志

#调用agent
response = agent_executor.invoke({'input': '5×6等于几？'})
print(response)



> Entering new AgentExecutor chain...
我应该使用乘法工具来计算两个整数的乘积。  
Action: multiply  
Action Input: "5 6"  30我已经知道答案  
Final Answer: 5×6等于30。

> Finished chain.
{'input': '5×6等于几？', 'output': '5×6等于30。'}


In [37]:
test_input = [
    "今天西安天气怎么样？",
    "9 * 13等于多少？",
    "电脑过热怎么办？"
]

for question in test_input:
    response = agent_executor.invoke({'input': question})
    print(response)



> Entering new AgentExecutor chain...
我需要查询西安的实时天气情况。  
Action: get_weather  
Action Input: "西安"  

SSLError: HTTPSConnectionPool(host='eolink.o.apispace.com', port=443): Max retries exceeded with url: /456456/weather/v001/now?areacode=101110101 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))